# Hackingtool environment helper

This notebook is ready for **Run All** in the repository's GitHub Codespace. The Codespace runs as `root` inside an isolated Linux container, so package installation is allowed without giving the notebook access to the host machine.

`Run All` performs environment detection and a permissions check. System package installation remains explicit: add package names to `packages_to_install` in the last cell when needed. Use security tools only on systems you own or are authorized to test.

In [ ]:
import os
import platform
import re
import shutil
import subprocess
from dataclasses import asdict, dataclass, field
from pathlib import Path


@dataclass
class OSInfo:
    system: str
    distro_id: str = ""
    distro_like: str = ""
    distro_version: str = ""
    pkg_manager: str = ""
    is_root: bool = False
    home_dir: Path = field(default_factory=Path.home)
    is_wsl: bool = False
    arch: str = ""


def detect() -> OSInfo:
    """Detect the operating system, Linux distribution, and package manager."""
    system = platform.system().lower()
    if system == "darwin":
        system = "macos"

    info = OSInfo(
        system=system,
        is_root=(os.geteuid() == 0) if hasattr(os, "geteuid") else False,
        home_dir=Path.home(),
        arch=platform.machine(),
    )

    if system == "linux":
        try:
            info.is_wsl = "microsoft" in Path("/proc/version").read_text(
                encoding="utf-8"
            ).lower()
        except (FileNotFoundError, PermissionError):
            pass

        os_release: dict[str, str] = {}
        for release_path in (Path("/etc/os-release"), Path("/usr/lib/os-release")):
            try:
                for line in release_path.read_text(encoding="utf-8").splitlines():
                    key, separator, value = line.partition("=")
                    if separator:
                        os_release[key.strip()] = value.strip().strip('"')
                break
            except (FileNotFoundError, PermissionError):
                continue

        info.distro_id = os_release.get("ID", "").lower()
        info.distro_like = os_release.get("ID_LIKE", "").lower()
        info.distro_version = os_release.get("VERSION_ID", "")

    for manager in ("apt-get", "pacman", "dnf", "zypper", "apk", "brew", "pkg"):
        if shutil.which(manager):
            info.pkg_manager = manager
            break

    return info


CURRENT_OS = detect()

PACKAGE_INSTALL_ARGS: dict[str, list[str]] = {
    "apt-get": ["apt-get", "install", "-y"],
    "pacman": ["pacman", "-S", "--noconfirm"],
    "dnf": ["dnf", "install", "-y"],
    "zypper": ["zypper", "install", "-y"],
    "apk": ["apk", "add"],
    "brew": ["brew", "install"],
    "pkg": ["pkg", "install", "-y"],
}

REQUIRED_PACKAGES: dict[str, list[str]] = {
    "apt-get": [
        "git", "python3-pip", "python3-venv", "curl", "wget",
        "ruby", "ruby-dev", "golang-go", "php", "default-jre-headless",
    ],
    "pacman": [
        "git", "python-pip", "curl", "wget", "ruby", "go", "php",
        "jre-openjdk-headless",
    ],
    "dnf": [
        "git", "python3-pip", "curl", "wget", "ruby", "golang", "php",
        "java-17-openjdk-headless",
    ],
    "zypper": ["git", "python3-pip", "curl", "wget", "ruby", "go", "php"],
    "brew": ["git", "python3", "curl", "wget", "ruby", "go", "php"],
    "pkg": ["git", "python3", "py39-pip", "curl", "wget", "ruby", "go", "php83"],
}

_PACKAGE_NAME = re.compile(r"^[A-Za-z0-9][A-Za-z0-9+_.:@/-]*$")


def install_packages(packages: list[str], os_info: OSInfo | None = None) -> bool:
    """Install explicitly requested system packages without invoking a shell."""
    info = os_info or CURRENT_OS
    if not packages:
        return True
    if info.pkg_manager not in PACKAGE_INSTALL_ARGS:
        raise RuntimeError(f"Unsupported package manager: {info.pkg_manager or 'none'}")

    invalid = [package for package in packages if not _PACKAGE_NAME.fullmatch(package)]
    if invalid:
        raise ValueError(f"Invalid package name(s): {invalid}")

    command = [*PACKAGE_INSTALL_ARGS[info.pkg_manager], *packages]
    if info.system == "linux" and not info.is_root:
        privilege_command = shutil.which("doas") or shutil.which("sudo")
        if not privilege_command:
            raise PermissionError("Root, doas, or sudo is required for package installation")
        command.insert(0, privilege_command)

    completed = subprocess.run(command, check=False)
    return completed.returncode == 0

In [ ]:
# Verify the runtime used by Run All.
runtime = asdict(CURRENT_OS)
runtime["home_dir"] = str(runtime["home_dir"])
for key, value in runtime.items():
    print(f"{key:16} {value}")

write_probe = Path.cwd() / ".hackingtool-notebook-write-check"
try:
    write_probe.write_text("ok", encoding="utf-8")
finally:
    write_probe.unlink(missing_ok=True)

if CURRENT_OS.system == "linux" and CURRENT_OS.is_root:
    print("\nReady: root permissions are available inside the isolated container.")
else:
    print("\nReady: notebook execution works; system installs may request elevation.")

In [ ]:
# Optional: add only the system packages you intend to install, then rerun this cell.
# Example: packages_to_install = ["git", "curl"]
packages_to_install: list[str] = []

if packages_to_install:
    if not install_packages(packages_to_install):
        raise RuntimeError("One or more packages failed to install")
    print("Installed:", ", ".join(packages_to_install))
else:
    print("No system packages requested; Run All is complete.")